# 面试问题：时态 RAG 怎样按“事实生效时间”检索，并处理冲突知识？

**一句话回答。** 将事实建模为带 valid-time 区间、发布时间、来源可信度和版本的记录；先按用户询问时点过滤，再对同一主谓的重叠事实检测冲突并拒答/升级，引用必须显示所用时间范围。不能把最新文档当成所有历史问题的答案。

本 Notebook 用 Python 标准库手写最小数据合同、状态机、验证器和失败分支。断言针对受控小数据，不等于模型语义正确、数据库安全、图谱质量或生产 Agent 的安全保证。

**资料入口。** [IA-RAG](https://arxiv.org/abs/2606.06044) 将动态知识表示为时间区间并在时间约束下检索；本例实现最小 valid-time 与冲突 gate。


In [ ]:
question = "Temporal RAG"  # 执行本行的状态、计算或校验逻辑。
assert "RAG" in question  # 执行本行的状态、计算或校验逻辑。
assert 10 - 4 == 6  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. 事实记录 valid time 与发布时间是不同坐标

valid time 表示事实在世界中何时生效，published time 表示系统何时知道它。历史问答主要过滤 valid time；回溯“当时系统知道什么”还需 transaction/published time，二者不可混用。


In [ ]:
facts = [{"id": "f1", "subject": "basic", "predicate": "price", "object": "10", "from": "2024-01-01", "to": "2024-06-30", "published": "2024-01-01", "source": "policy-v1", "trust": 2}, {"id": "f2", "subject": "basic", "predicate": "price", "object": "12", "from": "2024-07-01", "to": None, "published": "2024-07-01", "source": "policy-v2", "trust": 2}, {"id": "f3", "subject": "basic", "predicate": "price", "object": "9", "from": "2024-07-01", "to": None, "published": "2024-07-02", "source": "untrusted-note", "trust": 0}]  # 执行本行的状态、计算或校验逻辑。
assert len(facts) == 3  # 执行本行的状态、计算或校验逻辑。
assert facts[0]["to"] == "2024-06-30"  # 执行本行的状态、计算或校验逻辑。
assert facts[1]["published"] >= facts[1]["from"]  # 执行本行的状态、计算或校验逻辑。

## 2. 时间过滤优先于相关性排序

没有先过滤时间的向量检索会把最相似但已失效的政策送给模型。本例使用 ISO 日期字符串，生产应使用时区/精度明确的时间类型，并处理未知时间、开放区间和不同日历。


In [ ]:
def active_at(fact, moment):  # 执行本行的状态、计算或校验逻辑。
    return fact["from"] <= moment and (fact["to"] is None or moment <= fact["to"])  # 执行本行的状态、计算或校验逻辑。
assert active_at(facts[0], "2024-03-01")  # 执行本行的状态、计算或校验逻辑。
assert not active_at(facts[0], "2024-07-01")  # 执行本行的状态、计算或校验逻辑。
assert active_at(facts[1], "2025-01-01")  # 执行本行的状态、计算或校验逻辑。

## 3. 检索合同同时绑定主谓、时点和可信来源

查询解析可由 LLM 提出 subject/predicate/time 候选，但确定性层必须验证格式并执行时间/ACL/trust filter。对时间不明确的问题应追问或显式选择“当前”，不能悄悄假设用户意图。


In [ ]:
def retrieve(subject, predicate, moment, min_trust):  # 执行本行的状态、计算或校验逻辑。
    return [fact for fact in facts if fact["subject"] == subject and fact["predicate"] == predicate and fact["trust"] >= min_trust and active_at(fact, moment)]  # 执行本行的状态、计算或校验逻辑。
march = retrieve("basic", "price", "2024-03-01", 1)  # 执行本行的状态、计算或校验逻辑。
july = retrieve("basic", "price", "2024-07-01", 1)  # 执行本行的状态、计算或校验逻辑。
assert [fact["id"] for fact in march] == ["f1"]  # 执行本行的状态、计算或校验逻辑。
assert [fact["id"] for fact in july] == ["f2"]  # 执行本行的状态、计算或校验逻辑。
assert retrieve("basic", "price", "2024-07-01", 3) == []  # 执行本行的状态、计算或校验逻辑。

## 4. 冲突是检索结果，不应由模型任意投票

同一主谓、同一有效时间出现不同 object 时，要按来源权威、人工审核或业务规则处理；若没有确定规则则返回 conflict。把多个冲突句子同时塞进 prompt，常会让模型挑一个看似流畅的答案。


In [ ]:
def conflict(records):  # 执行本行的状态、计算或校验逻辑。
    return len({record["object"] for record in records}) > 1  # 执行本行的状态、计算或校验逻辑。
all_july = retrieve("basic", "price", "2024-07-01", 0)  # 执行本行的状态、计算或校验逻辑。
assert conflict(all_july)  # 执行本行的状态、计算或校验逻辑。
assert not conflict(july)  # 执行本行的状态、计算或校验逻辑。
assert {record["object"] for record in all_july} == {"9", "12"}  # 执行本行的状态、计算或校验逻辑。

## 5. 回答证据必须显示时间区间与来源版本

时间引用不能只给文档标题；用户需要知道该值在哪个 valid-time 区间适用。将 source/version、事实 id 和过滤时点写入 trace，也方便政策回溯与纠错。


In [ ]:
def citation(fact, moment):  # 执行本行的状态、计算或校验逻辑。
    return {"fact": fact["id"], "value": fact["object"], "valid": (fact["from"], fact["to"]), "source": fact["source"], "queried_at": moment}  # 执行本行的状态、计算或校验逻辑。
july_citation = citation(july[0], "2024-07-01")  # 执行本行的状态、计算或校验逻辑。
assert july_citation["value"] == "12"  # 执行本行的状态、计算或校验逻辑。
assert july_citation["valid"] == ("2024-07-01", None)  # 执行本行的状态、计算或校验逻辑。
assert july_citation["source"] == "policy-v2"  # 执行本行的状态、计算或校验逻辑。

## 6. 无事实与未来事实都应可解释地拒答

查询时间落在所有区间之外，不代表模型可以用参数记忆补全。回答需要区分“当前无有效记录”“时间歧义”“检测到冲突”和“权限不足”，以便上层决定澄清、人工审核或降级。


In [ ]:
def answer_state(records):  # 执行本行的状态、计算或校验逻辑。
    return "conflict" if conflict(records) else "answer" if records else "abstain"  # 执行本行的状态、计算或校验逻辑。
assert answer_state(march) == "answer"  # 执行本行的状态、计算或校验逻辑。
assert answer_state(all_july) == "conflict"  # 执行本行的状态、计算或校验逻辑。
assert answer_state(retrieve("basic", "price", "2023-01-01", 1)) == "abstain"  # 执行本行的状态、计算或校验逻辑。

## 7. 更新追加新 interval，而非覆写历史

政策变化应该关闭旧区间并追加新记录；覆写 f1 会使历史审计和离线评测无法重放。索引重建需要保留 tombstone/版本和更新时间，防止旧 embedding 或摘要混入新时态视图。


In [ ]:
def history(subject, predicate):  # 执行本行的状态、计算或校验逻辑。
    return [fact["id"] for fact in facts if fact["subject"] == subject and fact["predicate"] == predicate]  # 执行本行的状态、计算或校验逻辑。
assert history("basic", "price") == ["f1", "f2", "f3"]  # 执行本行的状态、计算或校验逻辑。
assert facts[0]["object"] == "10"  # 执行本行的状态、计算或校验逻辑。
assert facts[1]["object"] == "12"  # 执行本行的状态、计算或校验逻辑。

## 8. 评测按时点切片，而不是只报告当前准确率

建立有 gold valid-time、冲突与无答案标记的集合，报告 temporal recall、conflict precision、过期事实率、时间解析错误和引用区间正确率。当前政策的高分不能掩盖历史问答全部答错。


In [ ]:
def temporal_correct(records, expected):  # 执行本行的状态、计算或校验逻辑。
    return int(len(records) == 1 and records[0]["object"] == expected)  # 执行本行的状态、计算或校验逻辑。
assert temporal_correct(march, "10") == 1  # 执行本行的状态、计算或校验逻辑。
assert temporal_correct(july, "12") == 1  # 执行本行的状态、计算或校验逻辑。
assert temporal_correct(all_july, "12") == 0  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试回答要先要求用户/任务给出“问的是哪个时点”，再写 valid-time filter、冲突 gate、带区间的 citation、追加式更新和时态切片评测。时间过滤是检索正确性的前置条件，不是回答生成后的装饰字段。
